In [12]:
import os
from pathlib import Path
from dataclasses import dataclass
import urllib.request as request
import zipfile

from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, get_size
from cnnClassifier import logger

# 1. Anchor to the absolute project root ONCE
os.chdir(r"d:\chicken-disease-classification")
print(f"Current Working Directory set to: {os.getcwd()}")

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path 
    source_URL: str
    local_data_file: Path  
    unzip_dir: Path

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath) 
        
        create_directories([self.config.artifacts_root])
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        
        return data_ingestion_config

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename= self.config.local_data_file
            )
            logger.info(f"{filename} download! with follwing info: \n{headers}")
        else:
            logger.info(f"file already exists of size: {get_size(Path(self.config.local_data_file))}")
            
    def extract_zip_file(self):
        """
        extracts the zip file into the data directory
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            # Fixed: extractall instead of extractrall
            zip_ref.extractall(unzip_path) 

try: 
    config = ConfigurationManager() 
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
    print("✅ Data ingestion completed successfully!")
    
except Exception as e:
    logger.exception("Data ingestion failed")
    raise e

Current Working Directory set to: d:\chicken-disease-classification
[2026-06-01 12:47:11,848: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-01 12:47:11,852: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-01 12:47:11,856: INFO: common: created directory at: artifacts]
[2026-06-01 12:47:11,860: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-01 12:47:26,040: INFO: 4117094291: artifacts/data_ingestion/data.zip download! with follwing info: 
Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 2EBC:3CC99D:183C301:1D66F04:6A1D2CBA
Accept-Ranges: bytes
Date: Mon, 01 Jun 2026 07